In [27]:
from dotenv import load_dotenv 
from pathlib import Path 
import json 
import os

# ================================================================
# FILE PATH CONFIG
# ================================================================
load_dotenv()
BASE_PATH = os.environ.get("SOURCE_FOLDER")

if not BASE_PATH:
    raise ValueError("No SOURCE_FOLDER set in .env file")

ROOT_DIR = Path(BASE_PATH)
ANNO_PATH = ROOT_DIR/"notebooks"/"annos.json"

def load_json(path: Path)-> dict:
    with path.open("r", encoding='utf-8') as f:
        return json.load(f)


data = load_json(ANNO_PATH)
type(data)
data.keys()

dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])

In [8]:
# Check types 
images = data['images'][:3]
annos = data['annotations'][0]
type(annos['image_id'])

int

In [9]:
images[0].keys()
type(images[0]['id'])
images

[{'id': 0,
  'license': 1,
  'file_name': 'CarLongPlateGen566_jpg.rf.9e3b4294f915ef4e6b3981ec62eae463.jpg',
  'height': 303,
  'width': 472,
  'date_captured': '2026-01-26T09:42:31+00:00',
  'extra': {'name': 'CarLongPlateGen566.jpg'}},
 {'id': 1,
  'license': 1,
  'file_name': 'CarLongPlateGen596_jpg.rf.dcbc0a898b158955ed09124c707cd36d.jpg',
  'height': 300,
  'width': 468,
  'date_captured': '2026-01-26T09:42:31+00:00',
  'extra': {'name': 'CarLongPlateGen596.jpg'}},
 {'id': 2,
  'license': 1,
  'file_name': 'xemay1237_jpg.rf.d2bd4d01878a346cc34f27c64b192b3b.jpg',
  'height': 286,
  'width': 446,
  'date_captured': '2026-01-26T09:42:31+00:00',
  'extra': {'name': 'xemay1237.jpg'}}]

In [10]:
import pandas as pd

def select_n_images(n: int)-> list[str]:
    """
    n 
        number of images to be selected 
    """
    
    df = pd.read_csv('./selected_images/image_ranking.csv')
    df['image'] = df['image'].apply(lambda x: x[46:])
    
    return df[:n]['image'].to_list()
    

In [13]:
def filtered_imgs_annos(coco_json: dict, imgs_name_list: [str])-> list[dict]:
    """
    anno_data
        coco.json 
    img_list 
        list of images titles

    return 
        filtered coco.json containing only the img_list 
    """
    images = coco_json['images']
    annos = coco_json['annotations']

    filtered_annos = []
    filtered_imgs = []

    def make_img_id_list(imgs_name_l: [str])-> list[int]:
        img_ids = []
        for image in images:
            img_id = image['id']
            title = image['file_name']
            if title in imgs_name_l:
                img_ids.append(img_id)
                filtered_imgs.append(image)

        return img_ids

    img_ids = make_img_id_list(imgs_name_list)

    for anno in annos:
        img_id = anno['image_id']
        if img_id in img_ids:
            filtered_annos.append(anno)
            
    return filtered_imgs, filtered_annos
        

In [24]:
top_1000_imgs = select_n_images(1000)
filtered_1000_imgs, filtered_1000_annos = filtered_imgs_annos(data, top_1000_imgs)

type(filtered_1000_imgs)

list

In [29]:
# 'info', 'licenses', 'categories'
def create_filtered_coco_json(or_json: dict, filtered_n_imgs: list[dict], filtered_n_annos: list[dict]):
    info = or_json['info']
    license = or_json['licenses']
    categories = or_json['categories'][1]

    filtered_coco_json = {
        'info': info, 
        'licenses': license, 
        'categories': categories, 
        'images': filtered_n_imgs, 
        'annotations': filtered_n_annos
    }
    return filtered_coco_json


filtered_coco_json = create_filtered_coco_json(data, filtered_1000_imgs, filtered_1000_annos)



In [42]:
# CREATE a NEW JSON ON RT-DETR folder's dataset 
OUTPUT_FILE_PATH = ROOT_DIR/"RT-DETR"/"rtdetrv2_pytorch"/"dataset"/"

def make_coco_json_in_folder(output_file_path: path, json: dict):
    with open(output_file_path, 'w') as f:
        return json.dump(json, f)

make_coco_json_in_folder(OUTPUT_FILE_PATH, filtered_coco_json)

IsADirectoryError: [Errno 21] Is a directory: '/Users/boy/Desktop/car-parking-system/RT-DETR/rtdetrv2_pytorch/dataset'